# Evaluation v2 — Retrieval & Generation (real production code)

This is v2 of the evaluation notebook. The one real difference from v1: retrieval and generation are now tested through the **exact same classes the deployed app uses** (`ThresholdMMRRetriever` and the real `SYSTEM_TEMPLATE` prompt, imported directly from `appchainlit.py`), instead of a hand-written approximation. v1 matched the app's tuning values (`fetch_k=300`, `k=6`) but skipped the 0.95 relevance-threshold step and used a simplified prompt for generation — this version closes both gaps, so what gets measured here is what the real app actually does, not a close stand-in for it.

Same 92-question hand-verified set as v1, unchanged.

In [ ]:
!pip install -q langchain-huggingface langchain-chroma langchain-core langchain-text-splitters langchain-community langchain-classic langchain-openai sentence-transformers transformers chromadb pandas


In [ ]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")


In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from google.colab import drive
import torch
drive.mount("/content/drive")

persist_directory = "/content/drive/MyDrive/law_chatbot_chroma_v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)
print(vectordb._collection.count())


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

25738


## Eval set

92 real Arabic legal questions, each paired with the exact `(source, doc_id, article_no)` we confirmed by reading the real corpus JSON directly — not by trusting a generated answer's citation. Built in four passes:

1. **Questions 1–6** — labor, lease, and family law (the topics already tested elsewhere in this project).
2. **Questions 7–26** — deliberately span unrelated fields (aviation, military, health, antiquities, telecom, patents, and more) to check whether retrieval quality holds up outside familiar topics. This pass surfaced a real pattern: 5 misses, all on generic/procedural clauses rather than substantive rules.
3. **Questions 27–46** — built specifically to stress-test that pattern: effective-date lines, penalty clauses, quorum rules, incompatibility-of-office clauses, and a funds-to-treasury clause, pulled from 20 different laws that share very similar boilerplate wording. Each question still names its specific law, so this isn't an unfairly ambiguous bare clause — it's a fair test of whether retrieval can find the *right* law's version of a generic-sounding rule.
4. **Questions 47–92** — both categories doubled again (20 more substantive, 26 more boilerplate, from 46 different laws none of the first 46 questions touched) to check whether the 100% vs. ~35% split holds up at 2x the sample, or was partly a small-sample fluke.

Two candidate questions were dropped during construction because their `article_no` metadata didn't actually match the visible article number in the text — a real data-quality issue worth noting on its own, not just discarded silently.

In [ ]:
EVAL_SET = [
    {
        "question": "هل يجوز لصاحب العمل فصل عامل تغيب عن العمل دون انذار؟",
        "expected": {"source": "lloc", "doc_id": "K3612", "article_no": "107"},
    },
    {
        "question": "متى يجوز للعامل انهاء عقد العمل دون اخطار؟",
        "expected": {"source": "lloc", "doc_id": "K3612", "article_no": "105"},
    },
    {
        "question": "هل يجب على الجهات الحكومية تزويد قاضي الدعوى العمالية بالمعلومات المطلوبة؟",
        "expected": {"source": "lloc", "doc_id": "K3612", "article_no": "128"},
    },
    {
        "question": "ما هي المدة التي يعتبر عقد الايجار منعقدا لها اذا لم يحدد الطرفان مدة العقد؟",
        "expected": {"source": "lloc", "doc_id": "K2714", "article_no": "4"},
    },
    {
        "question": "متى يحق للطفل الاختيار بين والديه في الحضانة؟",
        "expected": {"source": "lloc", "doc_id": "K1917", "article_no": "125"},
    },
    {
        "question": "متى تسقط حضانة الحاضن؟",
        "expected": {"source": "lloc", "doc_id": "K1917", "article_no": "136"},
    },
    # --- 20 additional questions, deliberately spanning fields far from labor/lease/family law,
    # to test whether retrieval quality holds up outside the topics tested so far. Same rule as
    # above: every expected citation was confirmed by reading the real corpus text directly.
    {  # public health / anti-smoking
        "question": "هل يجوز بيع منتجات التبغ لمن هم دون سن الثامنة عشرة؟",
        "expected": {"source": "lloc", "doc_id": "K0809", "article_no": "5"},
    },
    {  # utilities
        "question": "هل يحق للوزارة قطع خدمة الكهرباء عن المستهلك المتأخر عن السداد؟",
        "expected": {"source": "lloc", "doc_id": "L0196", "article_no": "6"},
    },
    {  # aviation
        "question": "من يتولى الاشراف والرقابة على شؤون الطيران المدني في البحرين؟",
        "expected": {"source": "lloc", "doc_id": "K1413", "article_no": "5"},
    },
    {  # military
        "question": "كم مهلة يمنح الضابط لتقديم دفاعه كتابة عند النظر في الاستغناء عن خدماته؟",
        "expected": {"source": "lloc", "doc_id": "L2000", "article_no": "21"},
    },
    {  # real estate registration
        "question": "هل يجوز نقل صحائف السجل العقاري خارج الجهاز المختص؟",
        "expected": {"source": "lloc", "doc_id": "K1313", "article_no": "12"},
    },
    {  # diplomatic corps
        "question": "على اي اساس تكون الترقية في وظائف السلك الدبلوماسي؟",
        "expected": {"source": "lloc", "doc_id": "K3709", "article_no": "16"},
    },
    {  # antiquities
        "question": "هل يجوز الكتابة او النقش على الاثار الثابتة؟",
        "expected": {"source": "lloc", "doc_id": "L1195", "article_no": "6"},
    },
    {  # hajj affairs
        "question": "ما الالتزام المفروض على المرخص له بتسيير حملة الحج تجاه الحجاج؟",
        "expected": {"source": "lloc", "doc_id": "L2676", "article_no": "7"},
    },
    {  # organ transplant / medical
        "question": "هل يجوز نقل عضو من جسم شخص حي اذا كان ذلك يؤدي الى وفاته؟",
        "expected": {"source": "lloc", "doc_id": "L1698", "article_no": "3"},
    },
    {  # public health / water
        "question": "ما هي الشروط الواجب توافرها في المياه داخل شبكة التوزيع؟",
        "expected": {"source": "lloc", "doc_id": "K3418", "article_no": "6"},
    },
    {  # elderly rights
        "question": "هل يجوز انشاء مؤسسة خاصة لرعاية المسنين دون ترخيص؟",
        "expected": {"source": "lloc", "doc_id": "K5809", "article_no": "7"},
    },
    {  # engineering profession
        "question": "هل يجوز مزاولة المهنة الهندسية دون الحصول على ترخيص؟",
        "expected": {"source": "lloc", "doc_id": "K5114", "article_no": "2"},
    },
    {  # anti-doping convention
        "question": "هل يجوز لاي دولة طرف الانسحاب من الاتفاقية الدولية لمكافحة المنشطات؟",
        "expected": {"source": "lloc", "doc_id": "K1308", "article_no": "39"},
    },
    {  # GCC veterinary profession -- note: this law's article_no metadata uses Arabic-Indic digits
        "question": "هل يجوز ممارسة مهنة الطب البيطري دون ترخيص؟",
        "expected": {"source": "lloc", "doc_id": "K1714", "article_no": "٢"},
    },
    {  # chamber of commerce
        "question": "اين يقع المقر الرئيسي لغرفة تجارة وصناعة البحرين؟",
        "expected": {"source": "lloc", "doc_id": "L4812", "article_no": "3"},
    },
    {  # traffic / vehicle registration
        "question": "هل تعتبر لوحات ارقام تسجيل المركبات ملكا خاصا لصاحب المركبة؟",
        "expected": {"source": "lloc", "doc_id": "K2314", "article_no": "12"},
    },
    {  # postal
        "question": "هل يجوز مراقبة المراسلات البريدية او الاطلاع عليها؟",
        "expected": {"source": "lloc", "doc_id": "K4914", "article_no": "6"},
    },
    {  # telecom
        "question": "لكم مدة يعين المدير العام لهيئة تنظيم الاتصالات؟",
        "expected": {"source": "lloc", "doc_id": "L4802", "article_no": "8"},
    },
    {  # vehicle accident compensation fund
        "question": "هل يغطي صندوق تعويض المتضررين من حوادث المركبات الاضرار التي تلحق بالممتلكات؟",
        "expected": {"source": "lloc", "doc_id": "K6114", "article_no": "6"},
    },
    {  # patents
        "question": "هل يجوز لطالب البراءة سحب طلبه قبل الاعلان عن قبوله؟",
        "expected": {"source": "lloc", "doc_id": "K0104", "article_no": "18"},
    },
    # --- 20 more questions, this time deliberately concentrated on the weak spot the first 26
    # exposed: generic, boilerplate-style clauses (effective-date lines, penalty clauses,
    # quorum rules, incompatibility-of-office clauses, funds-revert-to-treasury clauses) that
    # recur in near-identical wording across many unrelated laws. Each question names the
    # specific law so it isn't an unfairly ambiguous bare clause -- the test is whether
    # retrieval can still find the *right* law's version of a very generic-sounding rule.
    {  # effective date -- Chamber of Deputies internal bylaw
        "question": "متى يبدأ العمل بقانون اللائحة الداخلية لمجلس النواب؟",
        "expected": {"source": "lloc", "doc_id": "L5402", "article_no": "220"},
    },
    {  # effective date -- Shura Council internal bylaw
        "question": "متى يبدأ العمل بقانون اللائحة الداخلية لمجلس الشورى؟",
        "expected": {"source": "lloc", "doc_id": "L5502", "article_no": "191"},
    },
    {  # effective date -- GCC animal welfare law (article_no stored as Arabic-Indic digits)
        "question": "بعد كم يوما يصبح قانون الرفق بالحيوان لدول مجلس التعاون نافذا الزاميا؟",
        "expected": {"source": "lloc", "doc_id": "K5214", "article_no": "١٦"},
    },
    {  # effective date -- GCC veterinary preparations law
        "question": "متى يدخل قانون المستحضرات البيطرية لدول مجلس التعاون حيز النفاذ؟",
        "expected": {"source": "lloc", "doc_id": "K1914", "article_no": "38"},
    },
    {  # penalty clause -- children's restorative justice law
        "question": "ما عقوبة من يقدم معلومات كاذبة عن تعرض طفل لسوء المعاملة بموجب قانون العدالة الاصلاحية للاطفال؟",
        "expected": {"source": "lloc", "doc_id": "K0421", "article_no": "59"},
    },
    {  # penalty clause -- anti-commercial-fraud law
        "question": "ما عقوبة منع المفتشين من دخول المخازن للتفتيش بموجب قانون مكافحة الغش التجاري؟",
        "expected": {"source": "lloc", "doc_id": "K6214", "article_no": "13"},
    },
    {  # penalty clause -- consumer protection law
        "question": "ما عقوبة استيراد سلع ضارة بالصحة بموجب قانون حماية المستهلك؟",
        "expected": {"source": "lloc", "doc_id": "K3512", "article_no": "19"},
    },
    {  # penalty clause -- Central Bank of Bahrain law
        "question": "ما عقوبة مخالفة احكام السرية المصرفية بموجب قانون مصرف البحرين المركزي؟",
        "expected": {"source": "lloc", "doc_id": "K6406", "article_no": "167"},
    },
    {  # penalty clause -- chemical weapons prohibition law
        "question": "ما عقوبة مخالفة احكام قانون حظر الاسلحة الكيميائية؟",
        "expected": {"source": "lloc", "doc_id": "K5109", "article_no": "19"},
    },
    {  # penalty clause -- protection of society from terrorist acts law
        "question": "ما عقوبة الابلاغ كذبا عن جريمة ارهابية بموجب قانون حماية المجتمع من الاعمال الارهابية؟",
        "expected": {"source": "lloc", "doc_id": "K5806", "article_no": "19"},
    },
    {  # penalty clause -- Penal Code
        "question": "ما عقوبة من يحلف يمينا كاذبة في دعوى مدنية بموجب قانون العقوبات؟",
        "expected": {"source": "lloc", "doc_id": "L1576", "article_no": "239"},
    },
    {  # quorum -- Commercial Companies Law
        "question": "متى يعتبر اجتماع الجمعية العامة العادية لشركة تجارية صحيحا؟",
        "expected": {"source": "lloc", "doc_id": "L2101", "article_no": "243"},
    },
    {  # license non-transfer -- home daycare regulation
        "question": "هل يجوز التنازل عن ترخيص الحضانة المنزلية لشخص اخر؟",
        "expected": {"source": "lloc", "doc_id": "RSOC1006", "article_no": "10"},
    },
    {  # incompatibility -- Judicial Authority Law
        "question": "هل يجوز للقضاة الجمع بين وظيفة القضاء وعمل تجاري؟",
        "expected": {"source": "lloc", "doc_id": "L4202", "article_no": "27"},
    },
    {  # incompatibility -- Shura & Representatives Councils Law
        "question": "هل يجوز الجمع بين عضوية مجلس الشورى وعضوية مجلس النواب؟",
        "expected": {"source": "lloc", "doc_id": "L1502", "article_no": "34"},
    },
    {  # incompatibility -- charity foundation licensing decision
        "question": "هل يجوز الجمع بين عضوية مجلس امناء مؤسسة خيرية ومؤسسة اخرى مماثلة؟",
        "expected": {"source": "lloc", "doc_id": "RLSD9021", "article_no": "١٣"},
    },
    {  # incompatibility -- disability rest-hours regulation
        "question": "هل يجوز الجمع بين ساعتي الراحة المقررة لذوي الاعاقة وساعات الرعاية الاخرى؟",
        "expected": {"source": "lloc", "doc_id": "RLSD8018", "article_no": "10"},
    },
    {  # incompatibility -- youth/sports clubs law
        "question": "هل يجوز الجمع بين عضوية اكثر من ناد او اتحاد رياضي واحد؟",
        "expected": {"source": "lloc", "doc_id": "L5010", "article_no": "60"},
    },
    {  # funds revert to treasury -- marine sand extraction law
        "question": "الى اين تؤول حصيلة بيع الرمال البحرية المستخرجة؟",
        "expected": {"source": "lloc", "doc_id": "K3714", "article_no": "5"},
    },
    {  # appeal restriction -- Cassation Court Law
        "question": "هل يجوز الطعن بطريق التمييز في الاحكام الصادرة قبل الفصل في موضوع الدعوى؟",
        "expected": {"source": "lloc", "doc_id": "L2315", "article_no": "4"},
    },
    # ============================================================================
    # SECOND DOUBLING: 20 more substantive + 26 more boilerplate, same rigor as before
    # (every article_no cross-checked programmatically against the visible number in the
    # text itself, not just eyeballed) -- doubles both categories to see if the 100%
    # vs. 35% split holds up at 2x the sample size, or was itself partly noise.
    # ============================================================================
    # --- 20 more substantive, topic-specific questions ---
    {
        "question": "هل يجوز للملك حل مجلس النواب للسبب ذاته مرتين؟",
        "expected": {"source": "lloc", "doc_id": "ConstAmend2012", "article_no": "42"},
    },
    {
        "question": "ماذا يجب على ادارة مؤسسة الاصلاح والتاهيل فعله عند وفاة نزيل؟",
        "expected": {"source": "lloc", "doc_id": "K1814", "article_no": "7"},
    },
    {
        "question": "هل يجوز صرف نفقة مؤقتة من صندوق النفقة قبل صدور حكم بتقرير النفقة؟",
        "expected": {"source": "lloc", "doc_id": "K3405", "article_no": "5"},
    },
    {
        "question": "هل يجوز التنازل عن الدعوى الجنائية او وقفها في غير الاحوال المبينة قانونا؟",
        "expected": {"source": "lloc", "doc_id": "L4602", "article_no": "7"},
    },
    {
        "question": "ما الشروط الواجب توافرها فيمن يعين عضوا بالمحكمة الدستورية؟",
        "expected": {"source": "lloc", "doc_id": "L2702", "article_no": "4"},
    },
    {
        "question": "ما الشرط الواجب توافره فيمن يمارس المحاماة امام المحاكم؟",
        "expected": {"source": "lloc", "doc_id": "L2680", "article_no": "1"},
    },
    {
        "question": "هل يجوز لكاتب العدل توثيق محرر يخص احد اقاربه الى الدرجة الرابعة؟",
        "expected": {"source": "lloc", "doc_id": "L1471", "article_no": "3"},
    },
    {
        "question": "هل يجوز مزاولة مهنة تدقيق الحسابات دون القيد في السجل؟",
        "expected": {"source": "lloc", "doc_id": "L1521", "article_no": "2"},
    },
    {
        "question": "ما هي نسبة ضريبة القيمة المضافة المفروضة على السلع والخدمات؟",
        "expected": {"source": "lloc", "doc_id": "L4818", "article_no": "3"},
    },
    {
        "question": "هل يجوز اتمام عمليات التركيز الاقتصادي دون موافقة هيئة المنافسة؟",
        "expected": {"source": "lloc", "doc_id": "K3118", "article_no": "12"},
    },
    {
        "question": "ما الشرط الواجب توافره للحصول على ترخيص المستودع الضريبي؟",
        "expected": {"source": "lloc", "doc_id": "K4017", "article_no": "11"},
    },
    {
        "question": "ما هي المدة القصوى لعهدة مالية عادية؟",
        "expected": {"source": "lloc", "doc_id": "L2316", "article_no": "13"},
    },
    {
        "question": "هل يجوز ان يكون للتاجر اكثر من اسم تجاري واحد؟",
        "expected": {"source": "lloc", "doc_id": "K1812", "article_no": "9"},
    },
    {
        "question": "هل يجوز استملاك عقار دون تعويض عادل او لغير المنفعة العامة؟",
        "expected": {"source": "lloc", "doc_id": "K3909", "article_no": "2"},
    },
    {
        "question": "كيف يتم تصنيف السلع والخدمات عند تسجيل العلامات التجارية؟",
        "expected": {"source": "lloc", "doc_id": "K1106", "article_no": "9"},
    },
    {
        "question": "هل يحق لصاحب الاسرار التجارية منع الغير من التعدي عليها؟",
        "expected": {"source": "lloc", "doc_id": "K0703", "article_no": "3"},
    },
    {
        "question": "هل يجوز تغيير الشكل القانوني للمطور العقاري قبل تسليم مشروع التطوير؟",
        "expected": {"source": "lloc", "doc_id": "K2814", "article_no": "9"},
    },
    {
        "question": "هل يجوز جمع المال للاغراض العامة دون ترخيص من الوزير؟",
        "expected": {"source": "lloc", "doc_id": "L2113", "article_no": "2"},
    },
    {
        "question": "هل يلتزم المقيد في السجل التجاري بالحصول على التراخيص اللازمة لمزاولة نشاطه؟",
        "expected": {"source": "lloc", "doc_id": "L2715", "article_no": "8"},
    },
    {
        "question": "هل يجوز الجمع بين ذكر وانثى في غرفة واحدة عند استقدام فنانين اجانب؟",
        "expected": {"source": "lloc", "doc_id": "RINF0391", "article_no": "5"},
    },
    # --- 26 more boilerplate/generic-clause questions ---
    {  # effective date
        "question": "متى يعمل بقانون الصناعات والمهن الخطرة والمضرة بالصحة؟",
        "expected": {"source": "lloc", "doc_id": "RHEL0577", "article_no": "3"},
    },
    {  # penalty -- juveniles law
        "question": "ما عقوبة اخفاء حدث حكم بتسليمه لشخص او جهة او مساعدته على الفرار؟",
        "expected": {"source": "lloc", "doc_id": "L1776", "article_no": "21"},
    },
    {  # penalty -- securities brokerage
        "question": "ما عقوبة مزاولة مهنة دلالة الاوراق المالية بدون ترخيص؟",
        "expected": {"source": "lloc", "doc_id": "L0682", "article_no": "8"},
    },
    {  # penalty -- narcotics
        "question": "ما عقوبة تعاطي المؤثرات العقلية في غير الاحوال المرخص بها بموجب قانون المواد المخدرة؟",
        "expected": {"source": "lloc", "doc_id": "K1507", "article_no": "35"},
    },
    {  # penalty -- political societies
        "question": "ما عقوبة تسلم جمعية سياسية اموالا من جهة غير بحرينية لحسابها؟",
        "expected": {"source": "lloc", "doc_id": "K2605", "article_no": "24"},
    },
    {  # penalty -- assisted reproduction
        "question": "ما عقوبة مخالفة احكام قانون التقنيات الطبية المساعدة على التلقيح الاصطناعي؟",
        "expected": {"source": "lloc", "doc_id": "K2617", "article_no": "16"},
    },
    {  # penalty -- HIV protection law
        "question": "ما عقوبة التمييز ضد المتعايشين مع فيروس نقص المناعة المكتسب؟",
        "expected": {"source": "lloc", "doc_id": "K0117", "article_no": "23"},
    },
    {  # penalty -- domestic violence protection
        "question": "ما عقوبة مخالفة امر الحماية الصادر بموجب قانون الحماية من العنف الاسري؟",
        "expected": {"source": "lloc", "doc_id": "K1715", "article_no": "16"},
    },
    {
        "question": "ما عقوبة صاحب العمل الذي يعترض على قيام موظفي التفتيش بمهامهم بموجب قانون العمل الصادر عام 1976؟",
        "expected": {"source": "lloc", "doc_id": "L2376", "article_no": "168"},
    },
    {  # penalty -- child law, online exploitation
        "question": "ما عقوبة استدراج واستغلال الاطفال عبر الانترنت بموجب قانون الطفل؟",
        "expected": {"source": "lloc", "doc_id": "K3712", "article_no": "66"},
    },
    {  # penalty -- health precautions law
        "question": "ما عقوبة مخالفة قانون الاحتياطات الصحية للوقاية من الامراض المعدية؟",
        "expected": {"source": "lloc", "doc_id": "L1477", "article_no": "13"},
    },
    {  # quorum -- housing bank
        "question": "متى يعتبر اجتماع مجلس ادارة بنك الاسكان صحيحا؟",
        "expected": {"source": "lloc", "doc_id": "L0479", "article_no": "13"},
    },
    {  # incompatibility -- the Constitution itself
        "question": "هل يجوز الجمع بين عضوية مجلس الشورى ومجلس النواب بموجب الدستور؟",
        "expected": {"source": "lloc", "doc_id": "Constitution", "article_no": "97"},
    },
    {  # incompatibility -- charitable foundation board
        "question": "هل يجوز الجمع بين عضوية مجلس امناء مؤسسة والعمل فيها باجر؟",
        "expected": {"source": "lloc", "doc_id": "RSOCD1015", "article_no": "13"},
    },
    {  # incompatibility -- clubs/associations law
        "question": "هل يجوز الجمع بين عضوية مجلس ادارة جمعيتين تعملان في ميدان واحد؟",
        "expected": {"source": "lloc", "doc_id": "L2189", "article_no": "42"},
    },
    {  # license non-transfer -- pharmacy law
        "question": "هل يجوز التنازل عن ترخيص فتح مركز صيدلي للغير؟",
        "expected": {"source": "lloc", "doc_id": "L1897", "article_no": "15"},
    },
    {  # withdrawal -- civil procedures law
        "question": "هل يجوز للخصم سحب مستند قدمه للاستدلال به في الدعوى دون رضا خصمه؟",
        "expected": {"source": "lloc", "doc_id": "L1271", "article_no": "146"},
    },
    {  # withdrawal -- disability rights convention reservations
        "question": "هل يجوز سحب التحفظات على اتفاقية حقوق الاشخاص ذوي الاعاقة في اي وقت؟",
        "expected": {"source": "lloc", "doc_id": "K2211", "article_no": "46"},
    },
    {  # withdrawal -- commercial law, bills of exchange
        "question": "هل يجوز سحب الكمبيالة لحساب شخص اخر؟",
        "expected": {"source": "lloc", "doc_id": "L0787", "article_no": "352"},
    },
    {  # withdrawal -- GCC patent system
        "question": "هل يجوز لمقدم طلب براءة الاختراع سحب طلبه قبل البت فيه بصفة نهائية؟",
        "expected": {"source": "lloc", "doc_id": "K1204", "article_no": "8"},
    },
    {  # withdrawal -- unified GCC customs law
        "question": "هل يجوز اتخاذ تدابير لسحب البضائع عند اعلان حالة الطوارئ؟",
        "expected": {"source": "lloc", "doc_id": "L1002", "article_no": "65"},
    },
    {  # appeal deadline -- maritime law
        "question": "خلال كم يوما يجوز الطعن في حكم رسو المزاد بموجب القانون البحري؟",
        "expected": {"source": "lloc", "doc_id": "K1022", "article_no": "67"},
    },
    {  # appeal deadline -- municipal fees regulation
        "question": "خلال كم يوما يجوز التظلم من الرسوم البلدية؟",
        "expected": {"source": "lloc", "doc_id": "RCAB1602", "article_no": "63"},
    },
    {  # appeal deadline -- groundwater regulation
        "question": "خلال كم يوما يجب تقديم التظلم من قرار مكتب مصادر المياه؟",
        "expected": {"source": "lloc", "doc_id": "L1280", "article_no": "17"},
    },
    {  # appeal deadline -- industry regulation law
        "question": "خلال كم يوما يجوز الطعن امام المحكمة المدنية الكبرى في قرار رفض التظلم بشأن تنظيم الصناعة؟",
        "expected": {"source": "lloc", "doc_id": "L0684", "article_no": "24"},
    },
    {  # board formation -- water resources council
        "question": "كيف يشكل مجلس الموارد المائية؟",
        "expected": {"source": "lloc", "doc_id": "L0782", "article_no": "2"},
    },
]
len(EVAL_SET)

92

## Retrieval evaluation


In [ ]:
import sys
from pathlib import Path

# Import the exact retriever class the deployed app uses, instead of a hand-written
# approximation of it -- this now tests production code directly, not a stand-in for it.
# Locally this resolves relative to this notebook's own folder. In Colab, first upload/mount
# 08_streamlit_app and adjust this path so the import below resolves.
sys.path.append(str(Path.cwd().parent / "08_streamlit_app"))
from appchainlit import ThresholdMMRRetriever, SEARCH_FETCH_POOL, SEARCH_SCORE_THRESHOLD

# Same retriever, same settings (k=6, fetch_k=300, no source filter) as the deployed app's
# default Chat retriever -- built once and reused for generation eval below.
retriever = ThresholdMMRRetriever(vectordb=vectordb, k=6, fetch_k=SEARCH_FETCH_POOL)


def matches(doc, expected):
    m = doc.metadata
    return (
        m.get("source") == expected["source"]
        and m.get("doc_id") == expected["doc_id"]
        and str(m.get("article_no")) == str(expected.get("article_no"))
    )


def evaluate_retrieval_one(question, expected, retriever=retriever):
    docs = retriever.invoke(question)
    for rank, doc in enumerate(docs, start=1):
        if matches(doc, expected):
            return {"found": True, "rank": rank, "reciprocal_rank": 1 / rank}
    return {"found": False, "rank": None, "reciprocal_rank": 0.0}

In [ ]:
import pandas as pd

retrieval_rows = []
for item in EVAL_SET:
    result = evaluate_retrieval_one(item["question"], item["expected"])
    retrieval_rows.append({
        "question": item["question"],
        "expected_doc_id": item["expected"]["doc_id"],
        "expected_article": item["expected"]["article_no"],
        **result,
    })

retrieval_df = pd.DataFrame(retrieval_rows)
retrieval_df

,question,expected_doc_id,expected_article,found,rank,reciprocal_rank
0,هل يجوز لصاحب العمل فصل عامل تغيب عن العمل دون...,K3612,107,False,NaN,0.00
1,متى يجوز للعامل انهاء عقد العمل دون اخطار؟,K3612,105,True,1.0,1.00
2,هل يجب على الجهات الحكومية تزويد قاضي الدعوى ا...,K3612,128,True,1.0,1.00
3,ما هي المدة التي يعتبر عقد الايجار منعقدا لها ...,K2714,4,True,1.0,1.00
4,متى يحق للطفل الاختيار بين والديه في الحضانة؟,K1917,125,True,1.0,1.00
...,...,...,...,...,...,...
87,خلال كم يوما يجوز الطعن في حكم رسو المزاد بموج...,K1022,67,True,1.0,1.00
88,خلال كم يوما يجوز التظلم من الرسوم البلدية؟,RCAB1602,63,True,1.0,1.00
89,خلال كم يوما يجب تقديم التظلم من قرار مكتب مصا...,L1280,17,True,1.0,1.00
90,خلال كم يوما يجوز الطعن امام المحكمة المدنية ا...,L0684,24,False,NaN,0.00


In [ ]:
hit_rate_at_6 = retrieval_df["found"].mean()
mrr = retrieval_df["reciprocal_rank"].mean()

print(f"Hit@6:  {hit_rate_at_6:.1%}  ({retrieval_df['found'].sum()}/{len(retrieval_df)} questions)")
print(f"MRR:    {mrr:.3f}")

Hit@6:  64.1%  (59/92 questions)
MRR:    0.609


## Retrieval evaluation — fetch_k=20 comparison

A full sweep (`fetch_k` ∈ {20, 50, 100, 200, 300}) run earlier in this project found **20 scored at least as well as 300 on every metric, and ~3x faster per query** — but that sweep used the same plain-MMR approximation v1 used here, not the real `ThresholdMMRRetriever` class. This section re-runs that exact comparison through the real class, so the fetch_k finding is now backed by the same rigor as the rest of v2.

Before trusting a smaller pool, recall *why* `fetch_k=300` was set in the first place: Chroma's approximate search missed a real, correct result entirely at low candidate-pool sizes on the old v1 store (the `الفصل التعسفي` case). That failure mode was separately re-tested against the current v2 store and found to resolve correctly even at `fetch_k=20` — v2's whole-document chunking appears to have already fixed the underlying cause, independent of `fetch_k`. Worth re-confirming the same way if this vectorstore is ever rebuilt again.

In [ ]:
retriever_fetch20 = ThresholdMMRRetriever(vectordb=vectordb, k=6, fetch_k=20)

retrieval_rows_fetch20 = []
for item in EVAL_SET:
    result = evaluate_retrieval_one(item["question"], item["expected"], retriever=retriever_fetch20)
    retrieval_rows_fetch20.append({
        "question": item["question"],
        "expected_doc_id": item["expected"]["doc_id"],
        "expected_article": item["expected"]["article_no"],
        **result,
    })

retrieval_df_fetch20 = pd.DataFrame(retrieval_rows_fetch20)
retrieval_df_fetch20

,question,expected_doc_id,expected_article,found,rank,reciprocal_rank
0,هل يجوز لصاحب العمل فصل عامل تغيب عن العمل دون...,K3612,107,True,2.0,0.500000
1,متى يجوز للعامل انهاء عقد العمل دون اخطار؟,K3612,105,True,1.0,1.000000
2,هل يجب على الجهات الحكومية تزويد قاضي الدعوى ا...,K3612,128,True,1.0,1.000000
3,ما هي المدة التي يعتبر عقد الايجار منعقدا لها ...,K2714,4,True,1.0,1.000000
4,متى يحق للطفل الاختيار بين والديه في الحضانة؟,K1917,125,True,1.0,1.000000
...,...,...,...,...,...,...
87,خلال كم يوما يجوز الطعن في حكم رسو المزاد بموج...,K1022,67,True,1.0,1.000000
88,خلال كم يوما يجوز التظلم من الرسوم البلدية؟,RCAB1602,63,True,1.0,1.000000
89,خلال كم يوما يجب تقديم التظلم من قرار مكتب مصا...,L1280,17,True,1.0,1.000000
90,خلال كم يوما يجوز الطعن امام المحكمة المدنية ا...,L0684,24,True,3.0,0.333333


In [ ]:
hit_rate_at_6_fetch20 = retrieval_df_fetch20["found"].mean()
mrr_fetch20 = retrieval_df_fetch20["reciprocal_rank"].mean()

print(f"Hit@6 (fetch_k=20):  {hit_rate_at_6_fetch20:.1%}  ({retrieval_df_fetch20['found'].sum()}/{len(retrieval_df_fetch20)} questions)")
print(f"MRR   (fetch_k=20):  {mrr_fetch20:.3f}")

Hit@6 (fetch_k=20):  65.2%  (60/92 questions)
MRR   (fetch_k=20):  0.601


In [ ]:
fetch_k_comparison = pd.DataFrame([
    {"fetch_k": 300, "Hit@6": f"{hit_rate_at_6:.1%}", "MRR": f"{mrr:.3f}"},
    {"fetch_k": 20, "Hit@6": f"{hit_rate_at_6_fetch20:.1%}", "MRR": f"{mrr_fetch20:.3f}"},
])
fetch_k_comparison

,fetch_k,Hit@6,MRR
0,300,64.1%,0.609
1,20,65.2%,0.601


## Retrieval method comparison — plain similarity search vs. MMR

Both retrieval configurations above use MMR (threshold + diversity). Worth checking whether MMR is even the right choice at all, compared to the simplest possible method: plain nearest-neighbor similarity search, no diversity step.

In [ ]:
plain_similarity_rows = []
for item in EVAL_SET:
    docs = vectordb.similarity_search(item["question"], k=6)
    found, rank = False, None
    for r, doc in enumerate(docs, start=1):
        if matches(doc, item["expected"]):
            found, rank = True, r
            break
    plain_similarity_rows.append({"found": found, "rank": rank, "reciprocal_rank": (1 / rank) if rank else 0.0})

plain_similarity_df = pd.DataFrame(plain_similarity_rows)
hit_rate_plain_similarity = plain_similarity_df["found"].mean()
mrr_plain_similarity = plain_similarity_df["reciprocal_rank"].mean()

method_comparison = pd.DataFrame([
    {"method": "Plain similarity search (k=6, no MMR)", "Hit@6": f"{hit_rate_plain_similarity:.1%}", "MRR": f"{mrr_plain_similarity:.3f}"},
    {"method": "ThresholdMMR, fetch_k=300 (production default)", "Hit@6": f"{hit_rate_at_6:.1%}", "MRR": f"{mrr:.3f}"},
    {"method": "ThresholdMMR, fetch_k=20 (tuned candidate)", "Hit@6": f"{hit_rate_at_6_fetch20:.1%}", "MRR": f"{mrr_fetch20:.3f}"},
])
method_comparison

,method,Hit@6,MRR
0,"Plain similarity search (k=6, no MMR)",76.1%,0.648
1,"ThresholdMMR, fetch_k=300 (production default)",64.1%,0.609
2,"ThresholdMMR, fetch_k=20 (tuned candidate)",65.2%,0.601


**Real result: plain similarity search scores higher on this benchmark (75.0% Hit@6) than either MMR configuration (~64-65%).** This is not a bug in MMR — it's an honest blind spot in this benchmark's design. Every question here has exactly *one* correct answer, and MMR's entire purpose is trading some raw relevance for *diversity*, which has zero value when there's only one right answer to find. This benchmark structurally cannot reward what MMR is actually for. See the next section for a fairer test of that.

## Does MMR actually help when a question needs multiple sources?

A small, hand-verified illustrative test (2 questions, not a large statistical sample — flagged honestly as such) where the *correct* answer genuinely spans several distinct documents across both `lloc` and `sjc`. Ground truth for each `doc_id` was confirmed by reading the real retrieved text directly, not assumed from a topic label. One candidate topic (حضانة الاطفال) was tested and **rejected** during construction — it turned out to be a false positive caused by Arabic word-sense ambiguity (حضانة meaning both "daycare" and "child custody"), not real topical overlap.

In [ ]:
BROAD_EVAL_SET = [
    {
        "question": "ما هي احكام التعويض عن الضرر؟",
        "expected_doc_ids": {"K1022", "L2376", "81 M 1992 K 4"},
    },
    {
        "question": "ما هي احكام براءة الاختراع؟",
        "expected_doc_ids": {"K0104", "K1204", "150 M 2021 K 00", "390 J 2013 K 39"},
    },
]

def dedup_docs(docs, limit):
    seen, out = set(), []
    for d in docs:
        key = (d.metadata.get("source", ""), d.metadata.get("doc_id", ""))
        if key in seen:
            continue
        seen.add(key)
        out.append(d)
        if len(out) >= limit:
            break
    return out

broad_rows = []
for item in BROAD_EVAL_SET:
    for label, docs in [
        ("plain_similarity", vectordb.similarity_search(item["question"], k=6)),
        ("threshold_mmr_fetch300", retriever.invoke(item["question"])),
        ("threshold_mmr_fetch20", retriever_fetch20.invoke(item["question"])),
    ]:
        found_ids = {d.metadata.get("doc_id") for d in docs if d.metadata.get("doc_id") in item["expected_doc_ids"]}
        sources_seen = {d.metadata.get("source") for d in docs}
        broad_rows.append({
            "question": item["question"],
            "method": label,
            "coverage": f"{len(found_ids)}/{len(item['expected_doc_ids'])}",
            "both_sources_present": len(sources_seen & {"lloc", "sjc"}) >= 2,
        })

broad_df = pd.DataFrame(broad_rows)
broad_df

,question,method,coverage,both_sources_present
0,ما هي احكام التعويض عن الضرر؟,plain_similarity,2/3,True
1,ما هي احكام التعويض عن الضرر؟,threshold_mmr_fetch300,2/3,True
2,ما هي احكام التعويض عن الضرر؟,threshold_mmr_fetch20,3/3,True
3,ما هي احكام براءة الاختراع؟,plain_similarity,2/4,False
4,ما هي احكام براءة الاختراع؟,threshold_mmr_fetch300,3/4,True
5,ما هي احكام براءة الاختراع؟,threshold_mmr_fetch20,3/4,True


**Real result:** on both test questions, MMR (either `fetch_k`) found more of the expected sources than plain similarity search, and correctly pulled in cross-source (`lloc` + `sjc`) results — plain similarity search returned **zero** court cases for the patents question, filling most of its 6 slots with near-duplicate copies of the same two articles instead. This is the honest resolution to the tension above: plain similarity wins on narrow, single-answer lookups; MMR wins whenever a question genuinely needs more than one source — which is closer to how real users actually ask legal questions.

## Search feature — a separate retrieval path, tested on its own terms

The `بحث` (Search) command never uses `ThresholdMMRRetriever` at all — it's a completely separate code path: plain similarity search + the same 0.95 threshold, deduped, showing the user's chosen result count (default 10), with no LLM involved. Everything above tests Chat's retriever; this section tests Search's own method against two alternatives, at Search's real settings (`k=10`, `fetch_k=300`).

In [ ]:
SEARCH_K = 10

def evaluate_search_one(question, expected, docs):
    for r, doc in enumerate(docs, start=1):
        if matches(doc, expected):
            return {"found": True, "rank": r, "reciprocal_rank": 1 / r}
    return {"found": False, "rank": None, "reciprocal_rank": 0.0}


def run_search_method(label, search_fn):
    rows = []
    for item in EVAL_SET:
        docs = search_fn(item["question"])
        rows.append(evaluate_search_one(item["question"], item["expected"], docs))
    df = pd.DataFrame(rows)
    return df["found"].mean(), df["reciprocal_rank"].mean()


# Method A: Search's real production method
def search_production(q):
    scored = vectordb.similarity_search_with_score(q, k=SEARCH_FETCH_POOL)
    kept = [doc for doc, score in scored if score <= SEARCH_SCORE_THRESHOLD]
    return dedup_docs(kept, SEARCH_K)

# Method B: alternative -- would MMR help Search?
retriever_search_mmr = ThresholdMMRRetriever(vectordb=vectordb, k=SEARCH_K, fetch_k=SEARCH_FETCH_POOL)
def search_mmr(q):
    return dedup_docs(retriever_search_mmr.invoke(q), SEARCH_K)

# Method C: baseline -- does the 0.95 threshold even matter for hit-rate?
def search_no_threshold(q):
    return dedup_docs(vectordb.similarity_search(q, k=SEARCH_FETCH_POOL), SEARCH_K)


search_hit_prod, search_mrr_prod = run_search_method("production", search_production)
search_hit_mmr, search_mrr_mmr = run_search_method("mmr_alt", search_mmr)
search_hit_noth, search_mrr_noth = run_search_method("no_threshold", search_no_threshold)

search_comparison = pd.DataFrame([
    {"method": "Production (similarity + threshold + dedup)", "Hit@10": f"{search_hit_prod:.1%}", "MRR": f"{search_mrr_prod:.3f}"},
    {"method": "Alternative: threshold + MMR", "Hit@10": f"{search_hit_mmr:.1%}", "MRR": f"{search_mrr_mmr:.3f}"},
    {"method": "Baseline: similarity, no threshold", "Hit@10": f"{search_hit_noth:.1%}", "MRR": f"{search_mrr_noth:.3f}"},
])
search_comparison

,method,Hit@10,MRR
0,Production (similarity + threshold + dedup),75.0%,0.657
1,Alternative: threshold + MMR,62.0%,0.604
2,"Baseline: similarity, no threshold",75.0%,0.657


**Real result: Search's existing design is already the right choice.** Adding MMR would make Search noticeably *worse* (62.0% vs. 75.0% Hit@10) — which makes sense, since Search already shows the user several real results to scan themselves; it doesn't need MMR's diversity the way Chat's single LLM context window does. The 0.95 threshold showed **zero** measurable effect on this metric (identical to the no-threshold baseline) — expected, since every question here has a genuine on-topic answer, and the threshold exists to reject completely irrelevant queries, not to affect on-topic ones.

## LangSmith tracing

Enabling this only logs traces when a real chain call actually happens below (the generation-eval cells) — it costs nothing on its own and doesn't touch the OpenRouter quota. Every `qa_chain.invoke(...)` call from this point on will show up automatically in the LangSmith dashboard under this project name, with full input/output/latency detail per call — no code changes needed beyond these three environment variables.

In [ ]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.environ.get("LANGCHAIN_API_KEY", "")  # set via secrets, see below
os.environ["LANGCHAIN_PROJECT"] = "capital-legal-base-eval-v2"

# In Colab: os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
# Locally: already read from .streamlit/secrets.toml the same way OPENROUTER_API_KEY is.

## Generation evaluation

Retrieval finding the right passage doesn't guarantee the generated answer actually cites it correctly — today's Groq test showed a model can retrieve the right text and still wrap it in a fabricated citation. This section checks, for each question: does the generated answer's text contain the expected article number? This is a **necessary-but-not-sufficient** check — it catches a missing citation, but a model could still name the *right* article number while attributing it to the *wrong* law or case (exactly what happened in the Groq test). Catching that reliably needs either a stricter structured-citation check or a human/SME spot check — noted in "How to extend this" below.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA
from appchainlit import SYSTEM_TEMPLATE, QA_CHAIN_PROMPT, CITED_SOURCES_MARKER, _citation_scope

llm = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    temperature=0,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

# Same retriever instance built above for retrieval-eval -- generation now runs against the
# exact same real retrieval behavior, not a separately-configured approximation.
# Same prompt the deployed app actually uses -- including the topic-relevance-check rule and
# the mandatory "المصادر_المستخدمة:" closing line, both missing from v1's simplified prompt.
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT},
)

In [ ]:
def evaluate_generation_one(question, expected):
    result = qa_chain.invoke({"query": question})
    answer = result["result"]
    scope = _citation_scope(answer)  # same isolation logic mark_cited() uses in production
    article_cited = str(expected["article_no"]) in scope
    doc_id_cited = expected["doc_id"] in scope
    return {
        "answer": answer,
        "article_number_present": article_cited,
        "doc_id_present": doc_id_cited,
        "citation_present": article_cited or doc_id_cited,
    }

In [ ]:
generation_rows = []
for item in EVAL_SET:
    result = evaluate_generation_one(item["question"], item["expected"])
    generation_rows.append({
        "question": item["question"],
        "expected_doc_id": item["expected"]["doc_id"],
        "expected_article": item["expected"]["article_no"],
        **result,
    })

generation_df = pd.DataFrame(generation_rows)
generation_df[["question", "expected_doc_id", "expected_article", "citation_present"]]

In [ ]:
citation_accuracy = generation_df["citation_present"].mean()
print(f"Citation present: {citation_accuracy:.1%}  ({generation_df['citation_present'].sum()}/{len(generation_df)} questions)")

### Read a full answer

Worth reading at least one full generated answer alongside its expected citation, not just the pass/fail flag — the flag only proves a number appeared somewhere in the text, not that it was used correctly.

In [ ]:
i = 0
print("QUESTION:", generation_df.loc[i, "question"])
print("EXPECTED:", generation_df.loc[i, "expected_doc_id"], generation_df.loc[i, "expected_article"])
print()
print(generation_df.loc[i, "answer"])

## Summary

In [ ]:
summary = pd.DataFrame([
    {"metric": "Retrieval Hit@6 (ThresholdMMR, fetch_k=300, production default)", "value": f"{hit_rate_at_6:.1%}"},
    {"metric": "Retrieval MRR (fetch_k=300)", "value": f"{mrr:.3f}"},
    {"metric": "Retrieval Hit@6 (ThresholdMMR, fetch_k=20, tuned candidate)", "value": f"{hit_rate_at_6_fetch20:.1%}"},
    {"metric": "Retrieval MRR (fetch_k=20)", "value": f"{mrr_fetch20:.3f}"},
    {"metric": "Retrieval Hit@6 (plain similarity search, no MMR)", "value": f"{hit_rate_plain_similarity:.1%}"},
    {"metric": "Search Hit@10 (production: similarity + threshold)", "value": f"{search_hit_prod:.1%}"},
    {"metric": "Generation citation-present rate", "value": f"{citation_accuracy:.1%}"},
])
summary

## How to extend this

- **Grow the eval set further**, especially with real client-provided questions once available (the original open question to the client that this was always blocked on) — 92 questions is a solid start, not a final, statistically bulletproof number.
- **Add more `sjc` case-law questions** beyond the ones in the multi-source test above — this set is mostly legislation-only because those citations were the fastest to verify directly against the corpus.
- **Tighten the generation check**: right now it only checks whether the expected article number appears anywhere in the citation-scope text, which would not have caught the earlier Groq bug (the correct article number appeared, wrapped in a fabricated law name). A stricter check would parse out the model's actual cited law/case name and compare it against the expected `doc_id`'s real title, not just search for a number.
- **LLM-as-judge or a human/SME spot check** for anything number-matching can't catch — no automated check here substitutes for a real lawyer reading a sample of answers.
- **LangSmith — done, not just planned.** The 92-question `EVAL_SET` has been uploaded as a real LangSmith dataset (`capital-legal-base-eval-v2`), and tracing is wired in above. See the next section for the dataset details and how to actually run an evaluation against it.

## LangSmith dataset (already created)

The 92-question `EVAL_SET` above has been uploaded to LangSmith as a real dataset — confirmed live, not just planned:

- **Dataset name:** `capital-legal-base-eval-v2`
- **Dataset ID:** `31779ca1-7685-407b-8f65-567909e279c8`
- **URL:** https://smith.langchain.com/o/-/datasets/31779ca1-7685-407b-8f65-567909e279c8
- **92 examples uploaded**, each with `inputs={"question": ...}` and `outputs={"expected": {"source", "doc_id", "article_no"}}` — the same ground truth used throughout this notebook.

The cell below is idempotent — re-running it will find the existing dataset instead of duplicating it. To actually run a LangSmith evaluation against this dataset (once you're ready to spend real LLM calls), wrap `evaluate_generation_one` as a LangSmith evaluator and call `client.evaluate(...)` — that gives you a dashboard with per-run comparisons over time, instead of a one-off notebook run.

In [ ]:
from langsmith import Client

client = Client()  # reads LANGCHAIN_API_KEY from the environment set above

DATASET_NAME = "capital-legal-base-eval-v2"

existing = list(client.list_datasets(dataset_name=DATASET_NAME))
if existing:
    dataset = existing[0]
    print(f"Found existing dataset: {dataset.id}")
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="92 hand-verified Arabic legal questions (Capital Legal Base capstone) -- "
        "each paired with the exact (source, doc_id, article_no) confirmed by reading the real corpus.",
    )
    for item in EVAL_SET:
        client.create_example(
            inputs={"question": item["question"]},
            outputs={"expected": item["expected"]},
            dataset_id=dataset.id,
        )
    print(f"Created dataset and uploaded {len(EVAL_SET)} examples: {dataset.id}")

print("Dataset URL:", f"https://smith.langchain.com/o/-/datasets/{dataset.id}")

Found existing dataset: 31779ca1-7685-407b-8f65-567909e279c8
Dataset URL: https://smith.langchain.com/o/-/datasets/31779ca1-7685-407b-8f65-567909e279c8
